# 01 · Simulation and data
**TraffiX** simulates central Tunis (Avenue Habib Bourguiba corridor) in SUMO. This notebook shows the
ingredients every agent works with: the configuration, the calendar that shapes demand (peaks, Ramadan,
Eids), the road network, one generated scenario, and the training data produced for the learning agents.

All code comes from the project package (`traffix/`); the notebook only calls it.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))          # the project root, so `traffix` can be imported
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)
from traffix.core import config
from traffix.evaluation import notebook_tools as nt

## 1. Configuration
Every assumption lives in YAML files under `config/`, so it can be changed without touching code.

In [ ]:
sim = config.simulation()
print("Base demand:", sim["demand"]["base_vehicles_per_hour"], "vehicles/h,",
      sim["demand"]["pedestrians_per_hour"], "pedestrians/h  ·  location:", sim["location"])
rows = [{"scenario": k, "starts": v["start"], "minutes": v["duration"] // 60,
         "hidden events": ", ".join(v.get("events", {})) or "-", "description": v.get("description", "")}
        for k, v in config.scenarios().items()]
pd.DataFrame(rows)

## 2. The calendar behind the demand
Demand = base × hour of day × weekday × month × calendar events. During **Ramadan** the daily profile
changes: a smaller, later morning peak, a **pre-iftar rush** before the computed sunset, then near-empty
streets during iftar. The agents never see these factors; they must learn them from observed traffic.

In [ ]:
from datetime import datetime, timedelta
from traffix.demand.calendar_model import CalendarModel
cal = CalendarModel()
def day_curve(day):
    times = [datetime.combine(day, datetime.min.time()) + timedelta(minutes=15 * i) for i in range(96)]
    vals = [cal.level(t) for t in times]
    level = [v[0] if isinstance(v, (tuple, list)) else v for v in vals]
    return [t.hour + t.minute / 60 for t in times], level
fig, ax = plt.subplots(figsize=(10, 3.5))
for day, label in [(datetime(2026, 10, 13).date(), "normal Tuesday, October 2026"),
                   (datetime(2026, 3, 5).date(), "Ramadan day, 5 March 2026")]:
    x, y = day_curve(day); ax.plot(x, y, label=label, lw=2)
ax.set_xlabel("hour of day"); ax.set_ylabel("traffic level (share of peak)"); ax.set_xticks(range(0, 25, 2))
ax.legend(); ax.grid(alpha=.3); ax.set_title("Hourly demand profile: normal day vs Ramadan"); plt.show()

In [ ]:
c = config.calendar()
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
months = c["month_factor"]
axes[0].bar(list(range(1, 13)), [months[m] if isinstance(months, list) else months[m] for m in (range(12) if isinstance(months, list) else range(1, 13))], color="#2bc4b5")
axes[0].set_title("Month factor"); axes[0].set_xticks(range(1, 13))
wd = c["weekday_factor"]
axes[1].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], [wd[i] for i in range(7)], color="#e6a23c")
axes[1].set_title("Weekday factor"); plt.tight_layout(); plt.show()

### Why train on 2025 and test on 2026
The Hijri calendar is about 11 days shorter than the Gregorian one, so Ramadan moves earlier every year.
A model that memorised "March is slow in the evening" would fail in 2026.

In [ ]:
from datetime import date
for year in (2025, 2026):
    days = [date(year, 1, 1) + timedelta(d) for d in range(366 if year % 4 == 0 else 365)]
    ram = [d for d in days if cal.info(datetime.combine(d, datetime.min.time()))["ramadan"]]
    print(f"Ramadan {year}: {ram[0]} to {ram[-1]} ({len(ram)} days)")

## 3. The road network
A schematic of central Tunis: Avenue Habib Bourguiba and Avenue Farhat Hached crossed by Rue de Rome,
Avenue de Carthage and Avenue Mohamed V, with 7 signalised junctions and bus stops on the two bus lines.

In [ ]:
import sumolib, xml.etree.ElementTree as ET
net_file = config.network_files()[0]
net = sumolib.net.readNet(net_file)
fig, ax = plt.subplots(figsize=(10, 6))
for e in net.getEdges():
    if e.getFunction(): continue
    xs, ys = zip(*e.getShape()); ax.plot(xs, ys, color="#555", lw=1 + e.getLaneNumber())
for n in net.getNodes():
    if n.getType() == "traffic_light":
        x, y = n.getCoord(); ax.scatter(x, y, s=300, color="#2bc4b5", zorder=3)
        ax.text(x, y, n.getID(), ha="center", va="center", fontsize=8, fontweight="bold", zorder=4)
stops = net_file.replace(".net.xml", ".stops.xml")
if os.path.exists(stops):
    from sumolib.geomhelper import positionAtShapeOffset
    for b in ET.parse(stops).getroot().findall("busStop"):
        lane = net.getLane(b.get("lane"))
        x, y = positionAtShapeOffset(lane.getShape(), float(b.get("endPos")))
        ax.scatter(x, y, marker="s", color="#1d4f9c", s=40, zorder=3)
ax.set_aspect("equal"); ax.axis("off"); ax.set_title("Network: junctions (teal) and bus stops (blue squares)"); plt.show()

## 4. One generated scenario
The demand generator turns the calendar into individual trips: cars, motorcycles, buses on their
lines with stops, pedestrians (including slow walkers), and the scenario's hidden events.

In [ ]:
name = nt.build_short("morning_peak", minutes=10)
root = ET.parse(os.path.join(nt.NB_ROUTES, f"{name}_seed1.rou.xml")).getroot()
types = pd.Series([v.get("type") for v in root.iter("vehicle")] + [v.get("type") for v in root.iter("trip")]).value_counts()
print("10 minutes of the morning peak (13 October 2026, 08:00):")
print(types.to_string()); print("pedestrians:", len(list(root.iter("person"))))

## 5. Training data
Training runs simulations on 2025 dates and records labelled examples. The labels come from the
simulation itself (the hidden incident, the injected sensor fault, the moment a vehicle really crossed),
so no human labelling was needed. Last column = what the model learns.

In [ ]:
train_dir = config.path("data", "training")
if os.path.isdir(train_dir):
    rows = []
    for f in sorted(os.listdir(train_dir)):
        d = pd.read_csv(os.path.join(train_dir, f))
        label = d.columns[-1]
        rows.append({"file": f, "rows": len(d), "inputs": len(d.columns) - 1, "label": label,
                     "label summary": f"{d[label].mean():.1%} positive" if set(d[label].dropna().unique()) <= {0, 1} else f"mean {d[label].mean():.2f}"})
    display(pd.DataFrame(rows))
    display(pd.read_csv(os.path.join(train_dir, "incident.csv")).head())
else:
    print("No training data yet: run  python scripts/train.py")